# Infográfico de orçamento para o keynote

Notebook source restored from its executed IPython history. It regenerates the locally retained 11.25 × 5.5 inch, 300 DPI figure. The figures are comparative context, not a claim that the budgets fund equivalent activities.

In [ ]:
"""Infografico SBGEA, 11.25 x 5.5 in, 300 DPI.

Executar: python gerar_infografico_orcamentos.py
Dependencias: matplotlib, numpy
Valores em milhoes de USD, cambio ilustrativo fixo USD 1 = BRL 5.
Fontes primarias e status de verificacao acompanham CADA registro.
Nao somar linhas: NASA Science esta contida em NASA total etc.
A comparacao e de escala institucional, NAO de despesas equivalentes em geofisica.
"""
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists()), None)
if ROOT is None: raise FileNotFoundError('Run this notebook inside the project checkout.')
sys.path.insert(0, str(ROOT / 'src')) if str(ROOT / 'src') not in sys.path else None
from sbgea_keynote.paths import output_path
import matplotlib.pyplot as plt
import numpy as np

FX_BRL_PER_USD = 5.0
DPI = 300
OUT = output_path('figures', 'keynote', 'infografico_11.25x5.5in.png')

# value = milhoes na moeda original; 'provisorio' indica fonte ainda nao
# auditada para a rubrica numerica exata, mesmo quando ha URL institucional.
DATA = [
 dict(label='NASA | Total', country='EUA', value=24838, currency='USD', year='2025', status='aprovado', source='CRS, Table 3, FY2025 enacted, NASA total', url='https://www.congress.gov/crs_external_products/R/PDF/R43419/R43419.115.pdf', verified=True),
 dict(label='NASA | Ciencia', country='EUA', value=7334.2, currency='USD', year='2025', status='aprovado', source='CRS, Table 3, FY2025 enacted, Science', url='https://www.congress.gov/crs_external_products/R/PDF/R43419/R43419.115.pdf', verified=True),
 dict(label='NASA | Heliofisica', country='EUA', value=805, currency='USD', year='2024', status='aprovado', source='NASA SMD FY2024 budget briefing', url='https://science.nasa.gov/wp-content/uploads/2024/04/hq-briefing-lpsc2024.pdf', verified=True),
 dict(label='NASA | Heliofisica (pedido)', country='EUA', value=787, currency='USD', year='2025', status='proposta', source='NASA FY2025 congressional justification, Heliophysics', url='https://www.nasa.gov/wp-content/uploads/2024/03/nasa-fy-2025-congressional-justification.pdf', verified=True),
 dict(label='NSF | Atmospheric & Geospace', country='EUA', value=293.8, currency='USD', year='2025', status='proposta; a conferir', source='NSF FY2025 Budget Request / GEO Division Funding', url='https://www.nsf.gov/about/budget/fy2025', verified=False),
 dict(label='NASA | Space Weather (pedido)', country='EUA', value=47.5, currency='USD', year='2025', status='proposta', source='NASA FY2025 congressional justification, Space Weather budget', url='https://www.nasa.gov/wp-content/uploads/2024/03/nasa-fy-2025-congressional-justification.pdf', verified=True),
 dict(label='CAPES | Dotacao final', country='Brasil', value=5730.251549, currency='BRL', year='2025', status='autorizado', source='CAPES: Informacoes Orcamentarias, Financeiras e Contabeis 2025', url='https://www.gov.br/capes/pt-br/acesso-a-informacao/transparencia-e-prestacao-de-contas/relatorios-de-gestao/processos-de-contas-anuais-2025/informacoes-orcamentarias-financeiras-e-contabeis', verified=True),
 dict(label='CNPq | Execucao (incl. parceiros)', country='Brasil', value=2800, currency='BRL', year='2025', status='executado', source='CNPq Relatorio de Gestao 2025, Resultados Alcancados', url='https://www.gov.br/cnpq/pt-br/acesso-a-informacao/auditorias/copy_of_RG2025vfinal31032026semlogo.pdf', verified=True),
 dict(label='AEB | Orcamento atualizado', country='Brasil', value=135.631669, currency='BRL', year='2024', status='autorizado', source='Portal da Transparencia, AEB 2024', url='https://portaldatransparencia.gov.br/orgaos/20402?ano=2024', verified=True),
 dict(label='INPE | Repasse previsto da AEB', country='Brasil', value=53.38, currency='BRL', year='2025', status='previsao; a conferir', source='Referencia previa a previsao de descentralizacao AEB-INPE; comprovante primario especifico pendente', url='https://www.gov.br/aeb/pt-br', verified=False),
]

plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'savefig.facecolor':'white'})
fig = plt.figure(figsize=(11.25,5.5), facecolor='white')
fig.text(.046,.855,'Escala de financiamento institucional  |  USD milhoes  |  R$ 5 = US$ 1',fontsize=10.6,color='#53627c')

# Dois blocos, MESMA escala x logaritmica. Barras sao sempre proporcionais
# aos valores convertidos, e as escalas tem os mesmos limites.
for country,rect,color in [ ('EUA',[.045,.245,.455,.495],'#0b3b83'),('Brasil',[.545,.245,.41,.495],'#139e87') ]:
    subset=[d for d in DATA if d['country']==country]
    ax=fig.add_axes(rect)
    y=np.arange(len(subset))[::-1]
    vals=np.array([d['value']/(FX_BRL_PER_USD if d['currency']=='BRL' else 1) for d in subset])
    # Barras proporcionais em escala linear, com eixo comum em USD 0..25000.
    ax.barh(y,vals,color=[color if d['verified'] else '#e0a03a' for d in subset],height=.47)
    ax.set_xlim(0,27000)
    ax.set_ylim(-.7,len(subset)-.2)
    ax.set_yticks([])
    ax.set_xticks([0,5000,10000,20000])
    ax.set_xticklabels(['0','5 bi','10 bi','20 bi'],fontsize=8,color='#69778b')
    ax.grid(axis='x',alpha=.12)
    ax.set_axisbelow(True)
    for sp in ax.spines.values():sp.set_visible(False)
    ax.tick_params(axis='x',length=0)
    fig.text(rect[0],.786,'ESTADOS UNIDOS' if country=='EUA' else 'BRASIL',fontsize=13,fontweight='bold',color=color)
    for i,(d,v) in enumerate(zip(subset,vals)):
        yy=y[i]
        lab=d['label']+(' *' if not d['verified'] else '')
        ax.text(0, yy+.28,lab,fontsize=8.8,color='#152b4a',va='bottom',weight='medium')
        pretty= f'US$ {v:,.1f} mi'.replace(',','X').replace('.',',').replace('X','.')
        # Value labels always outside bar unless very long bar.
        inside = v > 20000
        xpos = v-450 if inside else v+290
        ax.text(xpos,yy,pretty,fontsize=8.9,color='white' if inside else (color if d['verified'] else '#9a691b'),va='center',ha='right' if inside else 'left',weight='bold')
        ax.text(26800,yy,f"{d['year']}",ha='right',va='center',fontsize=7.4,color='#798595')

fig.text(.046,.157,'* Em amarelo: rubrica especifica ainda pendente de verificacao primaria.',fontsize=8.8,color='#835e26')
fig.text(.046,.119,'Aprovado, proposto e executado nao sao equivalentes. NASA Ciencia e Heliofisica estao dentro do total da NASA.',fontsize=8.7,color='#475875')
fig.text(.046,.084,'CAPES/CNPq cobrem muitas areas; INPE = apenas parcela prevista da AEB, nao orcamento total do instituto.',fontsize=8.7,color='#475875')
fig.text(.046,.049,'Fontes individuais (URLs, ano, rubrica, status) estao documentadas no script Python que gerou esta figura.',fontsize=8.4,color='#475875')
fig.savefig(OUT,dpi=DPI,facecolor='white')
print('Saved',OUT, 'pixels:',int(11.25*DPI),'x',int(5.5*DPI))